# 03 — Compile Benchmark Results

**Purpose:** Read the per-format, per-size metrics written to `/Volumes/{catalog}/{schema}/{volume}/artifacts/` and compile the full head-to-head comparison across the benchmark.

**Artifacts consumed:**

| Artifact | Source | Content |
|----------|--------|---------|
| `delta_{size}.json` | `01a_delta_native` | Write + ETL metrics (path-ref Delta) |
| `lance_{size}.json` | `01b_lance_native` | Write + ETL metrics (Lance native) |
| `delta_inline_{size}.json` | `01a_delta_native` (optional) | Write + ETL metrics (inline Delta anti-pattern) |
| `training_{size}.json` | `02_training_benchmark` | Training throughput: dummy + real, per format |
| `lance_convert_{size}.json` | `optional/01_lance_conversion` (optional) | Migration cost from existing files |

Each write/ETL artifact carries a normalized `common` block with identical keys so they stack into one table with no per-format key mapping. Training metrics are keyed by `formats[fmt][mode]`.

**Sections:**
1. Write & ETL comparison (the decisive Lance advantage)
2. Training throughput comparison (streaming read)
3. Consolidated summary matching the README key-highlights tables

In [0]:
# ── Widgets ─────────────────────────────────────────────────────────────────────
dbutils.widgets.dropdown("size", "10k", ["10k", "100k", "1m", "10m"], "Dataset size")
dbutils.widgets.text("catalog", "main", "UC catalog")
dbutils.widgets.text("schema", "ml_benchmark", "UC schema")
dbutils.widgets.text("volume", "lance_benchmark", "UC volume")

size    = dbutils.widgets.get("size")
catalog = dbutils.widgets.get("catalog")
schema  = dbutils.widgets.get("schema")
volume  = dbutils.widgets.get("volume")

base_vol      = f"/Volumes/{catalog}/{schema}/{volume}"
artifacts_dir = f"{base_vol}/artifacts"
print(f"Artifacts dir: {artifacts_dir}")
print(f"Size tier    : {size}")

In [0]:
import json, os
from pathlib import Path

def load_artifact(name):
    """Load a JSON artifact; returns None if it doesn't exist."""
    path = f"{artifacts_dir}/{name}"
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f)

# ── Write/ETL artifacts (from 01a, 01b, optional) ─────────────────────────────
delta_metrics  = load_artifact(f"delta_{size}.json")
lance_metrics  = load_artifact(f"lance_{size}.json")
inline_metrics = load_artifact(f"delta_inline_{size}.json")
convert_metrics = load_artifact(f"lance_convert_{size}.json")

# ── Training artifact (from 02) ─────────────────────────────────────────
training_metrics = load_artifact(f"training_{size}.json")

loaded = {k: v is not None for k, v in {
    "delta": delta_metrics, "lance": lance_metrics,
    "delta_inline": inline_metrics, "lance_convert": convert_metrics,
    "training": training_metrics,
}.items()}
print(f"Artifacts loaded: {loaded}")

In [0]:
import pandas as pd

# ── Section 1: Write & ETL comparison ───────────────────────────────────────
# Stack the common blocks from all write/ETL artifacts into one comparison table.
write_artifacts = [
    ("Lance (native)",      lance_metrics),
    ("Delta (path-ref)",    delta_metrics),
    ("Delta (inline)",      inline_metrics),
    ("Lance (conversion)",  convert_metrics),
]

rows = []
for label, m in write_artifacts:
    if m is None:
        continue
    c = m["common"]
    rows.append({
        "Format": label,
        "Write time (s)": c["write_total_s"],
        "Target write (s)": c["target_write_s"],
        "Output files": c["n_output_files"],
        "On-disk (GB)": round(c["on_disk_bytes"] / 1e9, 3),
        "ETL backfill (s)": c["etl_backfill_s"],
        "ETL bytes written (MB)": round(c["etl_bytes_written"] / 1e6, 1),
        "Round-trip OK": c["roundtrip_ok"],
    })

if rows:
    write_df = pd.DataFrame(rows)
    print(f"\n═══ WRITE & ETL COMPARISON — size={size} ═══")
    display(write_df)
else:
    print("No write/ETL artifacts found. Run 01a and/or 01b first.")

In [0]:
# ── Section 2: Training throughput comparison ───────────────────────────────
if training_metrics:
    fmt_data = training_metrics["formats"]
    train_rows = []
    for fmt, modes in fmt_data.items():
        for mode, metrics in modes.items():
            train_rows.append({
                "Format": fmt,
                "Mode": mode,
                "Samples/sec": metrics.get("samples_per_sec"),
                "Epoch wall (s)": metrics.get("epoch_wall_s"),
                "TTFB (s)": metrics.get("time_to_first_batch_s"),
                "Batch ms p50": metrics.get("batch_ms_p50"),
                "Batch ms p95": metrics.get("batch_ms_p95"),
                "Wait ms p50": metrics.get("wait_ms_p50"),
                "Wait ms p95": metrics.get("wait_ms_p95"),
            })

    if train_rows:
        train_df = pd.DataFrame(train_rows)
        print(f"\n═══ TRAINING THROUGHPUT — size={size}, "
              f"{training_metrics['num_gpu_workers']} GPU workers, "
              f"{training_metrics['num_epochs']} epochs ═══")
        display(train_df)

        # Relative throughput vs Lance (the baseline in the README highlights)
        lance_real = fmt_data.get("lance", {}).get("real", {})
        if lance_real:
            lance_sps = lance_real.get("samples_per_sec", 1)
            print("\n── Relative throughput (real mode, vs Lance baseline) ──")
            for fmt, modes in fmt_data.items():
                real = modes.get("real", {})
                if real and real.get("samples_per_sec"):
                    ratio = real["samples_per_sec"] / lance_sps
                    print(f"  {fmt:20s}: {ratio:.2f}x Lance  ({real['samples_per_sec']:.0f} samples/sec)")
else:
    print("No training artifact found. Run 02_training_benchmark first.")

In [0]:
# ── Section 3: Consolidated summary (README key-highlights format) ───────────
print(f"\n{'='*72}")
print(f"  BENCHMARK SUMMARY — size tier: {size}")
print(f"{'='*72}")

# Write scalability — render whatever write/ETL artifacts are present, rather than
# requiring the Lance + path-ref pair (an inline-only run has no delta_{size}.json).
_write_arms = [
    ("Lance",            lance_metrics),
    ("Delta (path-ref)", delta_metrics),
    ("Delta (inline)",   inline_metrics),
]
_present = [(label, m) for label, m in _write_arms if m is not None]
if _present:
    print(f"\n┌─ WRITE & ETL ───────────────────────")
    for label, m in _present:
        c = m["common"]
        print(f"│  {label:20s}: {c['write_total_s']:>7.1f}s  |  {c['n_output_files']:>6,} output files")
    print(f"│")
    print(f"│  ETL backfill (add column):")
    for label, m in _present:
        c = m["common"]
        note = " (new col only)" if label == "Lance" else (
            " (drags image bytes)" if label == "Delta (inline)" else "")
        print(f"│    {label:18s}: {c['etl_backfill_s']:>7.1f}s  |  {c['etl_bytes_written']/1e6:>8.1f} MB written{note}")
    print(f"└{'─'*60}")
else:
    print("\n(No write/ETL artifacts found — run 01a / 01b for this size.)")

# Training throughput
if training_metrics:
    fmt_data = training_metrics["formats"]
    print(f"\n┌─ TRAINING THROUGHPUT (streaming shuffle) ──────────────────────────")
    lance_real_sps = fmt_data.get("lance", {}).get("real", {}).get("samples_per_sec")
    for fmt in ["lance", "delta_inline", "delta"]:
        real = fmt_data.get(fmt, {}).get("real", {})
        dummy = fmt_data.get(fmt, {}).get("dummy", {})
        if not real:
            continue
        sps = real.get("samples_per_sec", 0)
        ttfb = real.get("time_to_first_batch_s", 0)
        ratio_str = ""
        if lance_real_sps and fmt != "lance":
            ratio = sps / lance_real_sps
            ratio_str = f"  ({ratio:.2f}x Lance)"
        print(f"│  {fmt:20s}: {sps:>8.0f} samples/sec  | TTFB {ttfb:.2f}s{ratio_str}")
    print(f"└{'─'*60}")

print(f"\nArtifacts dir: {artifacts_dir}")
print("Done.")

## Section 4 — Write & backfill scaling chart (across all tiers)

Unlike the sections above (which use the single `size` widget), this reads **every** tier's
artifacts so the *scaling trend* is visible — the whole point of the chart. Missing artifacts
are skipped, so path-ref (intentionally not run at 1M) simply stops at 100k.

- **Color = format** — blue Lance, Databricks-red Delta (inline), magenta Delta (path-ref)
- **Line style = operation** — solid write, dashed backfill
- **Log y-axis** so all series stay legible across the ~100× range

Saves `artifacts/write_backfill_scaling.png` (light mode, for the README).

In [ ]:
# ── Section 4: Write & backfill scaling chart (reads ALL tiers) ──────────────
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

TIERS       = ["10k", "100k", "1m"]        # x-axis, in scale order
TIER_LABELS = ["10k", "100k", "1M"]

# Validated CVD-safe palette (light mode); Databricks red for inline.
FORMATS = [                                # (short, legend label, artifact prefix, color)
    ("Lance",    "Lance",            "lance",        "#1E6FB4"),
    ("inline",   "Delta (inline)",   "delta_inline", "#FF3621"),
    ("path-ref", "Delta (path-ref)", "delta",        "#B84A9E"),
]
OPS = [                                     # (metric key, line style, op label)
    ("write_total_s",  "-",  "write"),
    ("etl_backfill_s", "--", "backfill"),
]

def _series(prefix, metric_key):
    """(xs, ys) for one format/metric across tiers; skips missing artifacts."""
    xs, ys = [], []
    for i, t in enumerate(TIERS):
        m = load_artifact(f"{prefix}_{t}.json")
        if m is not None:
            xs.append(i); ys.append(m["common"][metric_key])
    return xs, ys

fig, ax = plt.subplots(figsize=(8.2, 4.8), dpi=150)
fig.subplots_adjust(top=0.78)               # reserve a strip for title + two legend rows

for short, _lbl, prefix, color in FORMATS:
    for metric_key, style, op in OPS:
        xs, ys = _series(prefix, metric_key)
        if not xs:
            continue
        ax.plot(xs, ys, style, color=color, linewidth=2, marker="o", markersize=5.5,
                markeredgecolor="white", markeredgewidth=1.2, zorder=3)
        ax.annotate(f" {short} {op}", xy=(xs[-1], ys[-1]), va="center",
                    fontsize=8.5, color=color)

ax.set_yscale("log")
ax.set_yticks([2, 5, 10, 20, 50, 100, 200])
ax.set_yticklabels(["2s", "5s", "10s", "20s", "50s", "100s", "200s"])
ax.set_xticks(range(len(TIERS)))
ax.set_xticklabels(TIER_LABELS)
ax.set_xlim(-0.15, len(TIERS) - 1 + 0.95)   # right pad for direct labels
ax.set_xlabel("Dataset size (rows)")
ax.grid(axis="y", color="#e6e4df", linewidth=1)
for spine in ("top", "right", "left"):
    ax.spines[spine].set_visible(False)
ax.tick_params(length=0)

# Title on its own row above the legends.
fig.suptitle("Write & backfill scaling — Lance vs Delta", fontsize=13, fontweight="bold",
             x=0.09, ha="left", y=0.98)

# Two legend rows: color = format (upper), line style = operation (lower).
color_handles = [Line2D([0], [0], color=c, lw=2.5, label=lbl) for _s, lbl, _p, c in FORMATS]
style_handles = [Line2D([0], [0], color="#8a8780", lw=2, ls="-",  label="Write (solid)"),
                 Line2D([0], [0], color="#8a8780", lw=2, ls="--", label="Backfill (dashed)")]
leg1 = ax.legend(handles=color_handles, loc="lower left", frameon=False,
                 fontsize=9, ncol=3, bbox_to_anchor=(0, 1.13))
ax.add_artist(leg1)
ax.legend(handles=style_handles, loc="lower left", frameon=False,
          fontsize=9, ncol=2, bbox_to_anchor=(0, 1.02))

fig.text(0.09, 0.01, "Wall-clock seconds (log scale). Node-matched: 7 Ray vs 7 Spark workers. "
                     "Path-ref not run at 1M.", fontsize=8.5, color="#8a8780")

out_png = f"{artifacts_dir}/write_backfill_scaling.png"
fig.savefig(out_png, bbox_inches="tight", facecolor="white")
print(f"Saved {out_png}")
display(fig)